# Prática — Feature Engineering (Aula 3)
## Transformando as bases de eventos em uma tabela de modelagem

**O que vocês já têm:**
- O cadastro de colaboradores (`FtFuncionarioRH_amostra.csv`, Aula 2)
- O alvo já derivado (`alvo_derivado.csv`) — `pediu_para_sair`, construído a partir de `cIniciativaDemissao`
- As 5 bases de eventos da Aula 1 (`FtAbsenteismoMensalRH`, `FtAcidentesRH`, `FtHoraExtraRH`, `FtHorasIrregularesRH`, `FtMovimentoSalarialRH`)

**O que é novo hoje:**
- `datas_referencia.csv` — uma **data de referência por colaborador**. Ao construir qualquer métrica agregada, usem **apenas eventos até essa data** (inclusive). Isso não é opcional — é a mesma regra de janela temporal que vimos no material de apoio.

**Objetivo:** produzir uma única tabela, **uma linha por `nIdPessoa`**, juntando cadastro + alvo + métricas agregadas das 5 bases de eventos.

Dois exemplos abaixo já vêm resolvidos, como referência de padrão. As demais bases ficam para vocês.

## 0. Carregando os dados

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)

# Caminhos relativos ao diretório do notebook
# Notebook está em: aula3_material_recebido/notebook_pratica/
# Precisa voltar para: aula1_material_recebido/
# A1 = "../../../aula1/"
# A2 = "../../../aula2/"
# PRATICA = "./"

cadastro = pd.read_csv("..\\Bases de Dados\\FtFuncionarioRH_amostra.csv", sep=";")
alvo = pd.read_csv("..\\Bases de Dados\\alvo_derivado.csv", sep=";")
datas_ref = pd.read_csv("..\\Bases de Dados\\datas_referencia.csv", sep=";")
datas_ref['data_referencia'] = pd.to_datetime(datas_ref['data_referencia'])

print(cadastro.shape, alvo.shape, datas_ref.shape)
datas_ref.head()

(24000, 27) (24000, 2) (24000, 2)


,nIdPessoa,data_referencia
0,181076,2018-06-10
1,201073,2019-02-17
2,280267,2023-05-03
3,14686,2017-05-03
4,71442,2021-04-09


## 1. Função utilitária: filtrar eventos pela data de referência

Já está pronta — vocês vão usá-la em todas as bases.

In [4]:
cutoff_map = datas_ref.set_index('nIdPessoa')['data_referencia']

def filtra_por_data(df, col_data, dayfirst=True):
    """Mantem apenas linhas com data <= data de referencia do respectivo colaborador."""
    df = df.copy()
    df['__data'] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=dayfirst)
    df['__ref'] = df['nIdPessoa'].map(cutoff_map)
    df = df[df['__data'].notna() & df['__ref'].notna() & (df['__data'] <= df['__ref'])]
    return df.drop(columns=['__data', '__ref'])

def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "."), errors="coerce")

## 2. Exemplo resolvido — Acidentes

Coluna de data: `dDataAcidente`. Vamos gerar 3 métricas: número de eventos, quantos tiveram afastamento, e total de dias perdidos.

In [5]:
acid = pd.read_csv("..\\Bases de Dados\\FtAcidentesRH.csv", sep=";")
acid = filtra_por_data(acid, 'dDataAcidente')

acid['nComAfastamento_n'] = to_num(acid['nComAfastamento'])
acid['nDiasPerdidos_n'] = to_num(acid['nDiasPerdidos'])

agg_acid = acid.groupby('nIdPessoa').agg(
    acidentes_eventos=('nComAfastamento_n', 'count'),
    acidentes_com_afastamento=('nComAfastamento_n', 'sum'),
    acidentes_dias_perdidos=('nDiasPerdidos_n', 'sum')
).reset_index()

print(agg_acid.shape)
agg_acid.head()

(1986, 4)


,nIdPessoa,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos
0,11,2,1,15
1,128,1,1,4
2,265,1,1,90
3,676,1,1,30
4,1105,1,1,21


## 3. Exemplo resolvido — Movimentação Salarial

Coluna de data: `dMudanca`. Métricas: número de eventos, valor total, e **percentual médio** (média, não soma — percentual não se acumula da mesma forma que valor).

In [6]:
mov = pd.read_csv("..\\Bases de Dados\\FtMovimentoSalarialRH.csv", sep=";")
mov = filtra_por_data(mov, 'dMudanca')

mov['nValor_n'] = to_num(mov['nValor'])
mov['nPerc_n'] = to_num(mov['nPerc'])

agg_mov = mov.groupby('nIdPessoa').agg(
    mov_sal_eventos=('nValor_n', 'count'),
    mov_sal_valor_total=('nValor_n', 'sum'),
    mov_sal_perc_medio=('nPerc_n', 'mean')
).reset_index()

print(agg_mov.shape)
agg_mov.head()

(14540, 4)


,nIdPessoa,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio
0,11,8,838.003840,0.054387
1,31,2,211.252214,0.050400
2,57,6,558.894753,0.048567
3,71,5,296.058198,0.039080
4,73,3,262.274569,0.045633


## 4. Exercício 1 — Absenteísmo

Coluna de data: `dAnoMes`. Colunas disponíveis: `nQtdeAbsenteismo`, `nHoraPrevista`, `cTipo`, `nIdPessoa`.

Construam uma tabela `agg_abse` com uma linha por `nIdPessoa` e as colunas:
- `abs_eventos` — quantidade de eventos
- `abs_qtd_total` — soma de `nQtdeAbsenteismo`
- `horas_previstas_total` — soma de `nHoraPrevista`

Sigam o mesmo padrão dos exemplos acima (filtrar por data, converter para número, agregar).

In [7]:
abse = pd.read_csv("..\\Bases de Dados\\FtAbsenteismoMensalRH.csv", sep=";")

# Filtrar por data de referência (coluna dAnoMes)
abse = filtra_por_data(abse, 'dAnoMes')

# Converter as colunas numéricas relevantes
abse['nQtdeAbsenteismo_n'] = to_num(abse['nQtdeAbsenteismo'])
abse['nHoraPrevista_n'] = to_num(abse['nHoraPrevista'])

# Agregar por nIdPessoa -> agg_abse (abs_eventos, abs_qtd_total, horas_previstas_total)
agg_abse = abse.groupby('nIdPessoa').agg(
    abs_eventos=('nQtdeAbsenteismo_n', 'count'),
    abs_qtd_total=('nQtdeAbsenteismo_n', 'sum'),
    horas_previstas_total=('nHoraPrevista_n', 'sum')
).reset_index()

print(agg_abse.shape)
agg_abse.head()

C:\Users\mayumishimizu-ieg\AppData\Local\Temp\ipykernel_12280\863360617.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['__data'] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=dayfirst)


(62, 4)


,nIdPessoa,abs_eventos,abs_qtd_total,horas_previstas_total
0,11,7,9337.0,359037.0
1,31,1,1320.0,96622.0
2,34,0,0.0,13440.0
3,57,1,2200.0,225952.0
4,71,7,13521.0,224840.0


## 5. Exercício 2 — Hora Extra

Coluna de data: `dAnoMes`. Colunas disponíveis: `nReferencia`, `nValor`, `nIdPessoa`.

Construam `agg_hext` com:
- `he_eventos` — quantidade de eventos
- `he_referencia_total` — soma de `nReferencia`
- `he_valor_total` — soma de `nValor`

In [8]:
hext = pd.read_csv("..\\Bases de Dados\\FtHoraExtraRH.csv", sep=";")

# Filtrar por data de referência (coluna dAnoMes)
hext = filtra_por_data(hext, 'dAnoMes')

# Converter as colunas numéricas relevantes
hext['nReferencia_n'] = to_num(hext['nReferencia'])
hext['nValor_n'] = to_num(hext['nValor'])

# Agregar por nIdPessoa -> agg_hext (he_eventos, he_referencia_total, he_valor_total)
agg_hext = hext.groupby('nIdPessoa').agg(
    he_eventos=('nReferencia_n', 'count'),
    he_referencia_total=('nReferencia_n', 'sum'),
    he_valor_total=('nValor_n', 'sum')
).reset_index()

print(agg_hext.shape)
agg_hext.head()

C:\Users\mayumishimizu-ieg\AppData\Local\Temp\ipykernel_12280\863360617.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['__data'] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=dayfirst)


(8569, 4)


,nIdPessoa,he_eventos,he_referencia_total,he_valor_total
0,11,2,14.88,378.86
1,256,40,184.32,4018.37
2,356,36,191.52,4152.72
3,387,8,32.95,549.55
4,402,12,46.70,855.52


## 6. Exercício 3 — Horas Irregulares

Coluna de data: `dOcorrencia`. Colunas disponíveis: `nMinutosIrregularesExcedidos`, `nMinutosExtras`, `nIdPessoa`.

Construam `agg_hirr` com:
- `hi_eventos` — quantidade de eventos
- `hi_minutos_irregulares` — soma de `nMinutosIrregularesExcedidos`
- `hi_minutos_extras` — soma de `nMinutosExtras`

In [9]:
hirr = pd.read_csv("..\\Bases de Dados\\FtHorasIrregularesRH.csv", sep=";", low_memory=False)

# Filtrar por data de referência (coluna dOcorrencia)
hirr = filtra_por_data(hirr, 'dOcorrencia')

# Converter as colunas numéricas relevantes
hirr['nMinutosIrregularesExcedidos_n'] = to_num(hirr['nMinutosIrregularesExcedidos'])
hirr['nMinutosExtras_n'] = to_num(hirr['nMinutosExtras'])

# Agregar por nIdPessoa -> agg_hirr (hi_eventos, hi_minutos_irregulares, hi_minutos_extras)
agg_hirr = hirr.groupby('nIdPessoa').agg(
    hi_eventos=('nMinutosIrregularesExcedidos_n', 'count'),
    hi_minutos_irregulares=('nMinutosIrregularesExcedidos_n', 'sum'),
    hi_minutos_extras=('nMinutosExtras_n', 'sum')
).reset_index()

print(agg_hirr.shape)
agg_hirr.head()

(12391, 4)


,nIdPessoa,hi_eventos,hi_minutos_irregulares,hi_minutos_extras
0,11,2,893,0.0
1,31,73,10438,0.0
2,57,4,643,0.0
3,71,2,391,0.0
4,73,10,117,0.0


## 7. Juntando tudo

Esta parte já está pronta — só roda depois que `agg_abse`, `agg_hext` e `agg_hirr` estiverem criados acima.

In [10]:
df = cadastro.merge(alvo, on='nIdPessoa', how='left')
for agg in [agg_abse, agg_acid, agg_hext, agg_hirr, agg_mov]:
    df = df.merge(agg, on='nIdPessoa', how='left')

event_cols = ['abs_eventos','abs_qtd_total','horas_previstas_total','acidentes_eventos',
    'acidentes_com_afastamento','acidentes_dias_perdidos','he_eventos','he_referencia_total',
    'he_valor_total','hi_eventos','hi_minutos_irregulares','hi_minutos_extras',
    'mov_sal_eventos','mov_sal_valor_total','mov_sal_perc_medio']
df[event_cols] = df[event_cols].fillna(0)
df['pediu_para_sair'] = df['pediu_para_sair'].astype(int)

print("Shape final:", df.shape)
df.head()

Shape final: (24000, 43)


,nIdPessoa,nCodColigada,cFuncao,dAnoMes,EMPRESA,nCodFilial,cMes,nAno,cCargo,cSituacao,cSecao,nNrDependentes,cEstadoCivil,cSexo,cCor,cEscolaridade,cEstadoEndereco,cCidadeEndereco,cTurno,nTempoDeCasaAnos,nIdade,cGeracaoNascimento,cFaixasTempoDeCasa,cFaixasIdade,nSalarioTotal,nRemuneracaoTotal,cPosicaoFaixaSalarial,pediu_para_sair,abs_eventos,abs_qtd_total,horas_previstas_total,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,he_eventos,he_referencia_total,he_valor_total,hi_eventos,hi_minutos_irregulares,hi_minutos_extras,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio
0,181076,47,Operador de Produção I,set 2018,UBERABA - ABATE AVES,142,set,2018,Operação - Produção,Demitido,Gr Aves - Frango Inteiro - Alimentar Máq Pacot...,NaN,Solteiro,Masculino,Parda,Do 6º ao 9º ano do ensino fundamental,MG,Uberaba,1.0,"0,24",33,Geração Y,Até 3 Meses,De 31 a 35,1005,1005,Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000
1,201073,1,Ajudante de Serviços Gerais,mai 2019,CPG,4,mai,2019,Administração,Demitido,Limpeza Industrial - Higienização - 2º Turno,NaN,Solteiro,Feminino,Parda,Ensino médio completo,MS,Campo Grande,NaN,"0,24",32,Geração Y,Até 3 Meses,De 31 a 35,"1092,53","1092,53",Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000
2,280267,47,Operador de Produção I,ago 2023,FORQUILHINHA - ABATE AVES,24,ago,2023,Operação - Produção,Demitido,Gr Aves - Escald/Depen/Evisc - Evisceração - 2...,NaN,Solteiro,Masculino,Parda,Ensino médio completo,SC,Forquilhinha,2.0,"2,55",29,Geração Y,De 2 a 5 Anos,De 26 a 30,"1843,53","1843,53",80%,1,0.0,0.0,0.0,0.0,0.0,0.0,14.0,60.69,920.30,0.0,0.0,0.0,3.0,499.481131,0.111600
3,14686,1,Operador de Produção,ago 2017,CGR,77,ago,2017,Operação - Produção,Demitido,Embalagem Secundária - Encaixotamento - 1º Turno,NaN,Solteiro,Masculino,Parda,Ensino médio incompleto,MS,Campo Grande,NaN,"1,55",25,Geração Y,De 1 a 2 Anos,De 19 a 25,"1032,43","1032,43",Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,2.0,119.302130,0.063450
4,71442,47,Desossador de Coxa,jul 2021,FORQUILHINHA - ABATE AVES,24,jul,2021,Operação - Produção,Demitido,Gr Aves - Perna - Desossar Perna - 2° Turno,NaN,Solteiro,Masculino,Branca,Ensino fundamental completo,SC,Criciúma,2.0,"7,00",40,Geração X,De 5 a 10 Anos,De 36 a 45,"1637,81","1637,81",Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,33.91,595.53,3.0,1056.0,0.0,6.0,315.951223,0.036467


## 8. Conferência final

Antes de considerar pronto, respondam:
1. `df.shape[0]` bate com o número de colaboradores do cadastro?
2. A proporção de `pediu_para_sair` continua parecida com o que vimos na Aula 2?
3. Alguma das suas 3 colunas de `abs_eventos`, `he_eventos` ou `hi_eventos` parece grande ou pequena demais? Por quê?

In [11]:
print(df['pediu_para_sair'].value_counts(normalize=True))
print()
print(df[['abs_eventos','he_eventos','hi_eventos']].describe())

pediu_para_sair
1    0.513667
0    0.486333
Name: proportion, dtype: float64

        abs_eventos    he_eventos    hi_eventos
count  24000.000000  24000.000000  24000.000000
mean       0.005083      5.058125      8.950000
std        0.162968     11.091849     26.855846
min        0.000000      0.000000      0.000000
25%        0.000000      0.000000      0.000000
50%        0.000000      0.000000      1.000000
75%        0.000000      5.000000      6.000000
max       15.000000    109.000000    714.000000


In [12]:
pd.set_option('display.max_columns', None)
df.head()

,nIdPessoa,nCodColigada,cFuncao,dAnoMes,EMPRESA,nCodFilial,cMes,nAno,cCargo,cSituacao,cSecao,nNrDependentes,cEstadoCivil,cSexo,cCor,cEscolaridade,cEstadoEndereco,cCidadeEndereco,cTurno,nTempoDeCasaAnos,nIdade,cGeracaoNascimento,cFaixasTempoDeCasa,cFaixasIdade,nSalarioTotal,nRemuneracaoTotal,cPosicaoFaixaSalarial,pediu_para_sair,abs_eventos,abs_qtd_total,horas_previstas_total,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,he_eventos,he_referencia_total,he_valor_total,hi_eventos,hi_minutos_irregulares,hi_minutos_extras,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio
0,181076,47,Operador de Produção I,set 2018,UBERABA - ABATE AVES,142,set,2018,Operação - Produção,Demitido,Gr Aves - Frango Inteiro - Alimentar Máq Pacot...,NaN,Solteiro,Masculino,Parda,Do 6º ao 9º ano do ensino fundamental,MG,Uberaba,1.0,"0,24",33,Geração Y,Até 3 Meses,De 31 a 35,1005,1005,Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000
1,201073,1,Ajudante de Serviços Gerais,mai 2019,CPG,4,mai,2019,Administração,Demitido,Limpeza Industrial - Higienização - 2º Turno,NaN,Solteiro,Feminino,Parda,Ensino médio completo,MS,Campo Grande,NaN,"0,24",32,Geração Y,Até 3 Meses,De 31 a 35,"1092,53","1092,53",Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000
2,280267,47,Operador de Produção I,ago 2023,FORQUILHINHA - ABATE AVES,24,ago,2023,Operação - Produção,Demitido,Gr Aves - Escald/Depen/Evisc - Evisceração - 2...,NaN,Solteiro,Masculino,Parda,Ensino médio completo,SC,Forquilhinha,2.0,"2,55",29,Geração Y,De 2 a 5 Anos,De 26 a 30,"1843,53","1843,53",80%,1,0.0,0.0,0.0,0.0,0.0,0.0,14.0,60.69,920.30,0.0,0.0,0.0,3.0,499.481131,0.111600
3,14686,1,Operador de Produção,ago 2017,CGR,77,ago,2017,Operação - Produção,Demitido,Embalagem Secundária - Encaixotamento - 1º Turno,NaN,Solteiro,Masculino,Parda,Ensino médio incompleto,MS,Campo Grande,NaN,"1,55",25,Geração Y,De 1 a 2 Anos,De 19 a 25,"1032,43","1032,43",Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,2.0,119.302130,0.063450
4,71442,47,Desossador de Coxa,jul 2021,FORQUILHINHA - ABATE AVES,24,jul,2021,Operação - Produção,Demitido,Gr Aves - Perna - Desossar Perna - 2° Turno,NaN,Solteiro,Masculino,Branca,Ensino fundamental completo,SC,Criciúma,2.0,"7,00",40,Geração X,De 5 a 10 Anos,De 36 a 45,"1637,81","1637,81",Abaixo de 80%,0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,33.91,595.53,3.0,1056.0,0.0,6.0,315.951223,0.036467


## 9. Limpeza — Selecionando features para o modelo

O modelo de previsão de turnover usa:

**Features do cadastro (dados estáticos do colaborador):**
- `cargo` — posição hierárquica
- `empresa` — empresa/unidade
- `função` — área funcional
- `seção` — departamento/seção
- `faixas idade` — grupo etário
- `tempo de casa` — tempo de empresa
- `posicao faixa salarial` — posição na faixa salarial
- `salario total` — remuneração total

**Features de eventos (15 métricas agregadas):**
- Absenteísmo: `abs_eventos`, `abs_qtd_total`, `horas_previstas_total`
- Acidentes: `acidentes_eventos`, `acidentes_com_afastamento`, `acidentes_dias_perdidos`
- Hora Extra: `he_eventos`, `he_referencia_total`, `he_valor_total`
- Horas Irregulares: `hi_eventos`, `hi_minutos_irregulares`, `hi_minutos_extras`
- Movimentação Salarial: `mov_sal_eventos`, `mov_sal_valor_total`, `mov_sal_perc_medio`

**Alvo:**
- `pediu_para_sair` — indicador de saída da empresa

In [14]:
# Features do cadastro (dados estáticos)
features_cadastro = [
    'cargo', 'empresa', 'função', 'seção', 
    'faixas idade', 'tempo de casa', 'posicao faixa salarial', 'salario total'
]

# Definir as colunas que queremos manter (ID + features cadastro + features eventos + alvo)
colunas_modelo = ['nIdPessoa'] + features_cadastro + event_cols + ['pediu_para_sair']

# Verificar quais colunas existem realmente no dataframe
colunas_existentes = [col for col in colunas_modelo if col in df.columns]
colunas_faltantes = [col for col in colunas_modelo if col not in df.columns]

print(f"Colunas solicitadas: {len(colunas_modelo)}")
print(f"Colunas encontradas: {len(colunas_existentes)}")
print(f"Colunas faltantes: {colunas_faltantes}")

# Criar o dataframe com as colunas que existem
df_modelo = df[colunas_existentes].copy()

print(f"\nShape original: {df.shape}")
print(f"Shape modelo: {df_modelo.shape}")
print(f"\nColunas do modelo ({len(colunas_existentes)}):")
for col in colunas_existentes:
    print(f"  - {col}")

print(f"\nPrimeiras linhas:")
df_modelo.head()

Colunas solicitadas: 25
Colunas encontradas: 17
Colunas faltantes: ['cargo', 'empresa', 'função', 'seção', 'faixas idade', 'tempo de casa', 'posicao faixa salarial', 'salario total']

Shape original: (24000, 43)
Shape modelo: (24000, 17)

Colunas do modelo (17):
  - nIdPessoa
  - abs_eventos
  - abs_qtd_total
  - horas_previstas_total
  - acidentes_eventos
  - acidentes_com_afastamento
  - acidentes_dias_perdidos
  - he_eventos
  - he_referencia_total
  - he_valor_total
  - hi_eventos
  - hi_minutos_irregulares
  - hi_minutos_extras
  - mov_sal_eventos
  - mov_sal_valor_total
  - mov_sal_perc_medio
  - pediu_para_sair

Primeiras linhas:


,nIdPessoa,abs_eventos,abs_qtd_total,horas_previstas_total,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,he_eventos,he_referencia_total,he_valor_total,hi_eventos,hi_minutos_irregulares,hi_minutos_extras,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio,pediu_para_sair
0,181076,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0
1,201073,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0
2,280267,0.0,0.0,0.0,0.0,0.0,0.0,14.0,60.69,920.30,0.0,0.0,0.0,3.0,499.481131,0.111600,1
3,14686,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,2.0,119.302130,0.063450,0
4,71442,0.0,0.0,0.0,0.0,0.0,0.0,19.0,33.91,595.53,3.0,1056.0,0.0,6.0,315.951223,0.036467,0
